In [ ]:
from pipeline import run_retrieval_pipeline, get_final_context

query = "음주운전 면허취소 기준은?"

result = run_retrieval_pipeline(query)

In [ ]:
result["query_tokens"]

In [ ]:
result["reranked"][0]["metadata"]

In [ ]:
result["reranked"][0]["page_content"][:500]

In [ ]:
from pipeline import ask

answer = ask("어린이 보호구역 제한속도는 얼마인가?")
print(answer)

In [ ]:
import pandas as pd
from pipeline import run_full_pipeline

queries = [
    "어린이 보호구역에서는 몇 km로 달려야 해?",
    "스쿨존에서 속도위반하면 어떻게 돼?",
    "주정차 위반하면 과태료가 얼마나 나와?",
    "음주운전하면 면허가 바로 취소돼?",
    "불법주차한 차는 신고할 수 있어?",
    "주차위반한 차는 바로 견인될 수 있어?",
    "운전하면서 휴대폰 보면 처벌받아?",
    "음주측정 거부하면 어떻게 돼?",
    "횡단보도에서 보행자는 얼마나 보호받아?",
    "무면허운전하면 처벌이 어떻게 돼?"
]

summary_rows = []
doc_rows = []

for query in queries:
    result = run_full_pipeline(
        query=query,
        top_n=5,
        temperature=0.1,
        max_tokens=160
    )

    # 질문별 요약 저장
    summary_rows.append({
        "query": result["query"],
        "query_tokens": ", ".join(result["query_tokens"]),
        "answer": result["answer"],
        "num_final_docs": len(result["final_docs"]),
        "context_preview": result["context"][:1000]
    })

    # 선택된 문서별 상세 저장
    for doc in result["final_docs"]:
        doc_rows.append({
            "query": result["query"],
            "query_tokens": ", ".join(result["query_tokens"]),
            "final_rank": doc["rank"],
            "rerank_score": doc.get("rerank_score"),
            "rrf_score": doc.get("rrf_score"),
            "rrf_rank": doc.get("rrf_rank"),
            "dense_rank": doc.get("dense_rank"),
            "bm25_rank": doc.get("bm25_rank"),
            "metadata": str(doc["metadata"]),
            "page_content": doc["page_content"]
        })

df_summary = pd.DataFrame(summary_rows)
df_docs = pd.DataFrame(doc_rows)

# 전체 저장
df_summary.to_csv("file/final_test/final_pipeline_summary.csv", index=False, encoding="utf-8-sig")
df_docs.to_csv("file/final_test/final_pipeline_docs.csv", index=False, encoding="utf-8-sig")

# 질문별 개별 저장
for i, query in enumerate(queries, start=1):
    df_q_summary = df_summary[df_summary["query"] == query]
    df_q_docs = df_docs[df_docs["query"] == query]

    df_q_summary.to_csv(f"file/final_test/q{i}_summary.csv", index=False, encoding="utf-8-sig")
    df_q_docs.to_csv(f"file_final_test/q{i}_docs.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_docs.to_csv("file/final_test/final_pipeline_docs.csv", index=False, encoding="utf-8-sig")

# 질문별 개별 저장
for i, query in enumerate(queries, start=1):
    df_q_summary = df_summary[df_summary["query"] == query]
    df_q_docs = df_docs[df_docs["query"] == query]

    df_q_summary.to_csv(f"file/final_test/q{i}_summary.csv", index=False, encoding="utf-8-sig")
    df_q_docs.to_csv(f"file/final_test/q{i}_docs.csv", index=False, encoding="utf-8-sig")